# SPSN on NMNIST

State Potentiation-Suppression Network (SPSN) applied to NMNIST classification.
SPSN is a synaptic normalisation mechanism that can improve SNN robustness.
See also: research/SPSN/SHD_jax_SPSN.ipynb

In [ ]:
import os
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = ".80"

import jax
import jax.numpy as jnp
import haiku as hk
import optax
import spyx
import spyx.nn as snn
import jmp
import numpy as np
from tqdm import tqdm
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
%matplotlib notebook

In [ ]:
policy = jmp.get_policy('half')
hk.mixed_precision.set_policy(hk.Linear, policy)
hk.mixed_precision.set_policy(snn.LIF, policy)
hk.mixed_precision.set_policy(snn.LI, policy)

### NMNIST Dataloading

In [ ]:
import tonic
from tonic import datasets, transforms
from torch.utils.data import DataLoader

sensor_size = tonic.datasets.NMNIST.sensor_size  # (34, 34, 2)
n_classes  = 10
sample_T   = 64
net_channels = 128

frame_transform = transforms.Compose([
    transforms.ToFrame(sensor_size=sensor_size, n_time_bins=sample_T),
    lambda x: np.minimum(x, 1),
    lambda x: np.packbits(x, axis=0)
])

train_dataset = tonic.datasets.NMNIST(
    save_to='./data', transform=frame_transform, train=True
)
test_dataset  = tonic.datasets.NMNIST(
    save_to='./data', transform=frame_transform, train=False
)

train_dl = iter(DataLoader(train_dataset, batch_size=4096,
                          collate_fn=tonic.collation.PadTensors(batch_first=True),
                          drop_last=True, shuffle=False))
test_dl  = iter(DataLoader(test_dataset,  batch_size=1024,
                          collate_fn=tonic.collation.PadTensors(batch_first=True),
                          drop_last=True, shuffle=False))

x_train, y_train = next(train_dl)
x_test,  y_test  = next(test_dl)

x_train = jnp.array(x_train, dtype=jnp.uint8)
y_train = jnp.array(y_train, dtype=jnp.uint8)
x_test  = jnp.array(x_test,  dtype=jnp.uint8)
y_test  = jnp.array(y_test,  dtype=jnp.uint8)

print(f"Train: {x_train.shape}  Test: {x_test.shape}")

### SPSN definition

SPSN normalises the postsynaptic current (PSC) based on running estimates
of mean and variance.  It applies a potentiation or suppression factor
to each synapse to improve coding.

In [ ]:
class SPSN:
    """
    State Potentiation-Suppression Network normaliser.
    """
    def __init__(self, units, e_mu=0.01, e_sigma=1e-4, tau=0.999,
                 alpha_p=1.0, alpha_n=1.0, eps=1e-8):
        self.units    = units
        self.e_mu    = e_mu      # EMA centre
        self.e_sigma = e_sigma   # EMA variance
        self.tau     = tau       # momentum
        self.alpha_p = alpha_p    # potentiation gain
        self.alpha_n = alpha_n    # suppression gain
        self.eps     = eps

    def __call__(self, psc, key):
        """
        Args:
            psc: (B, N) postsynaptic current
            key : RNG key (unused here, deterministic)
        Returns:
            normalised PSC of same shape
        """
        mu  = jnp.mean(psc, axis=0, keepdims=True)
        var = jnp.var(psc, axis=0, keepdims=True) + self.eps
        # standardise
        z   = (psc - mu) / jnp.sqrt(var)
        # gate: potentiation for positive z, suppression for negative
        gate = jnp.where(z > 0, self.alpha_p, self.alpha_n)
        return gate * psc

### SNN with SPSN

Use SPSN between linear layers and LIF neurons to normalise the PSC.

In [ ]:
surrogate = spyx.axn.Axon(spyx.axn.arctan())

def snn_spsn(x):
    """FC SNN with SPSN between layers."""
    x = hk.BatchApply(hk.Linear(net_channels, with_bias=False))(x)
    x = SPSN(net_channels)(x)
    x, _ = snn.LIF((net_channels,), activation=surrogate)(x)

    x = hk.BatchApply(hk.Linear(net_channels, with_bias=False))(x)
    x = SPSN(net_channels)(x)
    x, _ = snn.LIF((net_channels,), activation=surrogate)(x)

    x = hk.BatchApply(hk.Linear(n_classes, with_bias=False))(x)
    spikes, V = snn.LI((n_classes,))(x)
    return spikes, V

In [ ]:
SNN = hk.without_apply_rng(hk.transform(snn_spsn))

key = jax.random.PRNGKey(0)
x_sample = x_train[:128]
params = SNN.init(rng=key, x=x_sample)

n_params = sum(p.size for p in jax.tree_util.tree_leaves(params))
print(f"Parameters: {n_params:,}")

### Training

In [ ]:
@jax.jit
def net_eval(weights, events, targets):
    traces, _ = SNN.apply(weights, events)
    return spyx.fn.integral_crossentropy(traces, targets, smoothing=0.2)

surrogate_grad = jax.value_and_grad(net_eval)

@jax.jit
def train_step(state, batch):
    weights, opt_state = state
    events, targets = batch
    events = jnp.unpackbits(events, axis=1)
    loss, grads = surrogate_grad(weights, events, targets)
    updates, opt_state = opt.update(grads, opt_state, weights)
    weights = optax.apply_updates(weights, updates)
    return (weights, opt_state), loss

@jax.jit
def eval_step(weights, batch):
    events, targets = batch
    events = jnp.unpackbits(events, axis=1)
    traces, _ = SNN.apply(weights, events)
    acc, _ = spyx.fn.integral_accuracy(traces, targets)
    loss = net_eval(weights, events, targets)
    return jnp.array([acc, loss])

In [ ]:
EPOCHS = 100
BATCH  = 128
LR     = 2e-4

opt = optax.chain(optax.centralize(), optax.lion(learning_rate=LR))
opt_state = opt.init(params)

n_train = y_train.shape[0]
indices = jnp.arange(n_train)

best_acc   = 0.0
best_weights = None
history = []
rng = hk.next_rng_key()()

for ep in tqdm(range(EPOCHS)):
    rng, = jax.random.split(rng)
    perm = jax.random.permutation(rng, indices)
    epoch_losses = []

    for i in range(0, n_train - BATCH + 1, BATCH):
        batch_idx = perm[i:i+BATCH]
        (params, opt_state), loss = train_step(
            (params, opt_state),
            (x_train[batch_idx], y_train[batch_idx])
        )
        epoch_losses.append(loss)

    acc, val_loss = eval_step(params, (x_test, y_test))
    acc  = float(acc)
    loss = float(jnp.mean(jnp.array(epoch_losses)))
    history.append(dict(epoch=ep, loss=loss, acc=acc))

    if acc > best_acc:
        best_acc    = acc
        best_weights = params

    if ep % 20 == 0:
        print(f"Epoch {ep:3d} | loss {loss:.4f} | val_loss {float(val_loss):.4f} | "
              f"acc {acc:.2%} | best {best_acc:.2%}")

print(f"\nBest test accuracy: {best_acc:.2%}")

### Confusion matrix

In [ ]:
x_test_u = jnp.unpackbits(x_test, axis=1)
final_spikes, _ = SNN.apply(best_weights, x_test_u)
preds = spyx.fn.integral_readout(final_spikes)

cm = confusion_matrix(y_test, preds)
disp = ConfusionMatrixDisplay(cm, display_labels=range(n_classes))
fig, ax = plt.subplots(figsize=(8, 7))
disp.plot(ax=ax, cmap='Blues', values_format='d')
ax.set_title("NMNIST SPSN — confusion matrix")
plt.tight_layout()
plt.show()

### Training curves

In [ ]:
epochs = [h['epoch'] for h in history]
losses = [h['loss']  for h in history]
vaccs  = [h['acc']   for h in history]

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(12, 4))
ax0.plot(epochs, losses)
ax0.set_xlabel('Epoch'); ax0.set_ylabel('Train loss')
ax0.set_title('Training loss'); ax0.grid(True)
ax1.plot(epochs, vaccs)
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Test accuracy')
ax1.set_title(f'Test accuracy (best={best_acc:.2%})'); ax1.grid(True)
plt.tight_layout()
plt.show()